# Graph Structural Evaluation

Supports both pipeline v5 and v6 output.

Six conditions, each evaluated **before** and **after** validation step:
- **A** — All relationships (hierarchy + directional + correlational + moderation)
- **B** — Merged corr+dir (correlational treated as directional)
- **C** — Without hierarchy (directional + correlational + moderation only)
- **D** — Hierarchy only
- **E** — Discard hierarchy lower nodes (remove lower-level nodes and all their edges)
- **F** — Type-agnostic (dir/corr/mod → 'edge'; hierarchy kept)

Metrics: GED, WL similarity, Spectral similarity, Edge F1 (P/R/F1), SHD/|E_GT|


In [ ]:
# Cell 1 — Imports

import json
import re
import os
import signal
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import networkx as nx
from numpy.linalg import eigvalsh

warnings.filterwarnings("ignore")
print("Imports OK.")


In [ ]:
# Cell 2 — Load data
# Point PRED_CSV_PATH to whichever pipeline output you want to evaluate.
# The post-validation column prefix is auto-detected.

GT_JSON_PATH  = "<path_to_data>"

# ── Choose one ──────────────────────────────────────────────────────────────
PRED_CSV_PATH = "../data/output/pipeline_v9_batch/pipeline_v4_results_330version.csv"
OUTPUT_DIR    = "../data/output/pipeline_v9_eval"

# PRED_CSV_PATH = "../data/output/pipeline_v4_output/pipeline_v4_results_validated.csv"
# OUTPUT_DIR    = "../data/output/pipeline_v4_output"

# PRED_CSV_PATH = "../data/output/pipeline_v5_output/pipeline_v5_results_validated.csv"
# OUTPUT_DIR    = "../data/output/pipeline_v5_output"
# ────────────────────────────────────────────────────────────────────────────

with open(GT_JSON_PATH, "r") as f:
    gt_articles = json.load(f)

# Try utf-8 first, fall back to latin-1 for files with non-ASCII characters
try:
    pred_df = pd.read_csv(PRED_CSV_PATH, encoding="utf-8")
except UnicodeDecodeError:
    pred_df = pd.read_csv(PRED_CSV_PATH, encoding="latin-1")
    print("Note: loaded with latin-1 encoding")
gt_by_id = {a["id"]: a for a in gt_articles}

pred_ids    = [int(str(aid).replace("json_", "")) for aid in pred_df["article_id"]]
pred_df["num_id"] = pred_ids
overlap_ids = sorted(set(pred_ids) & set(gt_by_id.keys()))

print(f"CSV               : {PRED_CSV_PATH}")
print(f"Ground truth      : {len(gt_articles)} articles")
print(f"Predicted         : {len(pred_df)} articles")
print(f"Overlap           : {len(overlap_ids)} articles")


In [ ]:
# Cell 3 — Parse ground truth → MultiDiGraph
#
# Relation label → edge_type:
#   hierarchy          → "hierarchy"
#   direction__*       → "directional"
#   correlation__*     → "correlational"  (single directed edge)
#   moderation__*      → "moderation"
#
# A single relation can carry MULTIPLE labels (e.g. moderation + correlation).
# We loop over all labels per relation.

def _edge_type(label: str) -> str:
    if label == "hierarchy":            return "hierarchy"
    if label.startswith("direction"):   return "directional"
    if label.startswith("correlation"): return "correlational"
    if label.startswith("moderation"):  return "moderation"
    return "unknown"

def parse_gt_graph(article: dict) -> nx.MultiDiGraph:
    G   = nx.MultiDiGraph()
    ann = article["annotations"][0]
    vm  = {}
    for r in ann["result"]:
        if r.get("type") == "labels" and "Variable" in r["value"].get("labels", []):
            vm[r["id"]] = r["value"].get("text", "").strip()
    for r in ann["result"]:
        if r.get("type") == "textarea" and r.get("from_name") == "var_name" and r["id"] in vm:
            nn = r["value"].get("text", "")
            if isinstance(nn, list): nn = nn[0] if nn else ""
            if nn.strip(): vm[r["id"]] = nn.strip()
    for rid, name in vm.items():
        G.add_node(rid, label=name)
    for r in ann["result"]:
        if r.get("type") == "relation":
            fid, tid = r.get("from_id", ""), r.get("to_id", "")
            if fid in vm and tid in vm:
                # Loop over ALL labels (a relation can have multiple)
                for lbl in r.get("labels", []):
                    G.add_edge(fid, tid, edge_type=_edge_type(lbl), raw_label=lbl)
    return G

gt_graphs = {aid: parse_gt_graph(gt_by_id[aid]) for aid in overlap_ids}

print("GT graph summary:")
print(f"{'ID':>4} {'Nodes':>6} {'Edges':>6}  {'hier':>5} {'dir':>5} {'corr':>5} {'mod':>5}")
for aid in overlap_ids:
    G  = gt_graphs[aid]
    et = Counter(d["edge_type"] for _,_,d in G.edges(data=True))
    print(f"{aid:>4} {G.number_of_nodes():>6} {G.number_of_edges():>6}  "
          f"{et.get('hierarchy',0):>5} {et.get('directional',0):>5} "
          f"{et.get('correlational',0):>5} {et.get('moderation',0):>5}")


In [ ]:
# Cell 4 — Parse pipeline CSV → MultiDiGraph (pre and post validation)
#
# Pre  columns: step2_hierarchy + step4_*         (consistent across all pipelines)
# Post columns: auto-detected from what exists in the CSV:
#   pp_*       (v6 post-processed)
#   step5_2_*  (v5 step 5.2)
#   step5_1_*  (fallback)

def _safe(val) -> str:
    return "" if pd.isna(val) else str(val).strip()


def _smart_split(text: str, delim: str = ";") -> list:
    """
    Split a semicolon-delimited string, but rejoin fragments that were
    broken inside parentheses.  Handles variable names like:
        Drug-testing program punitiveness (punitive vs less punitive; highly punitive)
    which contain a literal semicolon inside parentheses.

    Returns a list of properly reassembled parts.
    """
    raw_parts = text.split(delim)
    result = []
    buf = ""
    depth = 0
    for part in raw_parts:
        if buf:
            buf += delim + part
        else:
            buf = part
        # Track depth from this part only (incremental)
        depth += part.count("(") - part.count(")")
        if depth <= 0:
            result.append(buf.strip())
            buf = ""
            depth = 0
    if buf:  # trailing unbalanced — flush anyway
        result.append(buf.strip())
    return result


# Auto-detect post-validation columns.
# Different pipeline outputs use different column naming conventions:
#   5_1_1_hierarchy / 5_1_1_direction / 5_1_1_correlation / 5_1_1_moderation  (Copy v4)
#   pp_hierarchy / pp_directional / pp_correlational / pp_moderation          (v6)
#   step5_2_hierarchy / step5_2_directional / ...                             (v4/v5)
#   step5_1_hierarchy / step5_1_directional / ...                             (fallback)
#
# We detect the prefix and store a column-name map for the four edge types.

_cols = set(pred_df.columns)

if "step5_1_hierarchy" in _cols:
    # Manual-revision columns (direction/correlation not directional/correlational)
    _POST_COL_MAP = {
        "hierarchy":     "step5_1_hierarchy",
        "directional":   "step5_1_directional",
        "correlational": "step5_1_correlational",
        "moderation":    "step5_1_moderation",
    }
    _POST_LABEL = "step5_1"
else:
    _POST_COL_MAP = None
    _POST_LABEL = None
    print("WARNING: no post-validation columns found — post graphs will mirror pre.")

print(f"Post-validation columns: {_POST_LABEL!r}")
if _POST_COL_MAP:
    for k, v in _POST_COL_MAP.items():
        present = v in _cols
        print(f"  {k:15s} → {v:30s} {'✓' if present else '✗ MISSING'}")


def parse_pred_graph(row: pd.Series, prefix: str) -> nx.MultiDiGraph:
    """
    Parse one predicted graph from a CSV row.
    prefix='pre'  → uses step2_hierarchy + step4_*
    prefix='post' → uses auto-detected post columns
    """
    G = nx.MultiDiGraph()
    _seen = set()

    def _ensure(n):
        if n and n not in G: G.add_node(n, label=n)

    def _add_edge(s, t, et):
        key = (s, t, et)
        if key not in _seen:
            G.add_edge(s, t, edge_type=et)
            _seen.add(key)

    # Nodes
    for v in _smart_split(_safe(row.get("step2_final_vars", ""))):
        v = v.strip()
        if v: G.add_node(v, label=v)

    if prefix == "pre":
        hier_col = "step2_hierarchy"
        dir_col  = "step4_directional"
        corr_col = "step4_correlational"
        mod_col  = "step4_moderation"
    else:  # post
        if _POST_COL_MAP:
            hier_col = _POST_COL_MAP["hierarchy"]
            dir_col  = _POST_COL_MAP["directional"]
            corr_col = _POST_COL_MAP["correlational"]
            mod_col  = _POST_COL_MAP["moderation"]
        else:  # fallback to pre
            hier_col = "step2_hierarchy"
            dir_col  = "step4_directional"
            corr_col = "step4_correlational"
            mod_col  = "step4_moderation"
        # hierarchy may be in step2 even for post if post col doesn't exist
        if hier_col not in _cols:
            hier_col = "step2_hierarchy"

    # Hierarchy
    for e in _smart_split(_safe(row.get(hier_col, ""))):
        e = e.strip()
        if "->" in e:
            p = e.split("->")
            if len(p) == 2:
                s, t = p[0].strip(), p[1].strip()
                _ensure(s); _ensure(t)
                if s and t: _add_edge(s, t, "hierarchy")

    # Directional
    for e in _smart_split(_safe(row.get(dir_col, ""))):
        e = e.strip()
        m = re.search(r'\[(validated|null|hypothesized)\]$', e)
        if m: e = e[:m.start()].strip()
        if "->" in e:
            p = e.split("->")
            if len(p) == 2:
                s, t = p[0].strip(), p[1].strip()
                _ensure(s); _ensure(t)
                if s and t: _add_edge(s, t, "directional")

    # Correlational
    for e in _smart_split(_safe(row.get(corr_col, ""))):
        e = e.strip()
        m = re.search(r'\[(validated|null|hypothesized)\]$', e)
        if m: e = e[:m.start()].strip()
        if "<->" in e:
            p = e.split("<->")
            if len(p) == 2:
                v1, v2 = p[0].strip(), p[1].strip()
                _ensure(v1); _ensure(v2)
                if v1 and v2: _add_edge(v1, v2, "correlational")

    # Moderation
    for e in _smart_split(_safe(row.get(mod_col, ""))):
        e = e.strip()
        if not e: continue
        m = re.search(r'\[(validated|null|hypothesized)\]$', e)
        if m: e = e[:m.start()].strip()
        mm = re.match(r'(.+?)\s+moderates\s+\((.+)\)', e)
        if mm:
            mod = mm.group(1).strip()
            # Use _smart_split with "," to handle commas inside parentheses
            for mv in _smart_split(mm.group(2), delim=","):
                mv = mv.strip()
                _ensure(mod); _ensure(mv)
                if mv: _add_edge(mod, mv, "moderation")
    return G


pred_graphs_pre  = {}
pred_graphs_post = {}
for _, row in pred_df.iterrows():
    aid = row["num_id"]
    if aid in set(overlap_ids):
        pred_graphs_pre[aid]  = parse_pred_graph(row, "pre")
        pred_graphs_post[aid] = parse_pred_graph(row, "post")

def _graph_summary(graphs, label):
    print(f"\n{label}:")
    print(f"{'ID':>4} {'Nodes':>6} {'Edges':>6}  {'hier':>5} {'dir':>5} {'corr':>5} {'mod':>5}")
    for aid in overlap_ids:
        G  = graphs[aid]
        et = Counter(d["edge_type"] for _,_,d in G.edges(data=True))
        print(f"{aid:>4} {G.number_of_nodes():>6} {G.number_of_edges():>6}  "
              f"{et.get('hierarchy',0):>5} {et.get('directional',0):>5} "
              f"{et.get('correlational',0):>5} {et.get('moderation',0):>5}")

_graph_summary(pred_graphs_pre,  "Predicted graphs — PRE  validation")
_graph_summary(pred_graphs_post, "Predicted graphs — POST validation")


In [ ]:
# Cell 5 — Graph helpers

def filter_graph(G: nx.MultiDiGraph, exclude_types=()) -> nx.MultiDiGraph:
    """Copy of G with specified edge types removed."""
    H = nx.MultiDiGraph()
    for n, d in G.nodes(data=True): H.add_node(n, **d)
    for u, v, d in G.edges(data=True):
        if d.get("edge_type") not in exclude_types:
            H.add_edge(u, v, **d)
    return H

def keep_only(G: nx.MultiDiGraph, keep_types) -> nx.MultiDiGraph:
    """Copy of G keeping only edges whose edge_type is in keep_types."""
    H = nx.MultiDiGraph()
    for n, d in G.nodes(data=True): H.add_node(n, **d)
    for u, v, d in G.edges(data=True):
        if d.get("edge_type") in keep_types:
            H.add_edge(u, v, **d)
    return H

def remap_corr_to_dir(G: nx.MultiDiGraph) -> nx.MultiDiGraph:
    """Copy of G where every 'correlational' edge is relabelled 'directional'."""
    H = nx.MultiDiGraph()
    for n, d in G.nodes(data=True): H.add_node(n, **d)
    for u, v, d in G.edges(data=True):
        nd = dict(d)
        if nd.get("edge_type") == "correlational":
            nd["edge_type"] = "directional"
        H.add_edge(u, v, **nd)
    return H


def discard_hierarchy_lower_nodes(G: nx.MultiDiGraph) -> nx.MultiDiGraph:
    """
    Remove all lower-level nodes (targets of hierarchy edges) and every
    edge incident to them.  Keeps upper-level hierarchy nodes and all
    non-hierarchy nodes.  The hierarchy edges themselves are also removed
    (their target was a lower node).
    """
    lower = {v for u, v, d in G.edges(data=True)
             if d.get("edge_type") == "hierarchy"}
    H = nx.MultiDiGraph()
    for n, d in G.nodes(data=True):
        if n not in lower:
            H.add_node(n, **d)
    for u, v, d in G.edges(data=True):
        if u not in lower and v not in lower:
            H.add_edge(u, v, **d)
    return H


def propagate_to_upper(G: nx.MultiDiGraph) -> nx.MultiDiGraph:
    """
    For each hierarchy edge (upper -> lower) in G, copy every non-hierarchy
    edge incident to 'lower' onto 'upper':
      - lower -> X  becomes  upper -> X  (same edge_type)
      - X -> lower  becomes  X -> upper  (same edge_type)

    Applied iteratively until convergence, so multi-level hierarchies
    (A -> B -> C) are handled correctly.

    Applied to BOTH GT and pred as a preprocessing step.
    """
    H = G.copy()
    while True:
        added = 0
        hier_pairs = [(u, v) for u, v, d in H.edges(data=True)
                      if d.get("edge_type") == "hierarchy"]
        existing = {(u, v, d.get("edge_type")) for u, v, d in H.edges(data=True)}

        for upper, lower in hier_pairs:
            for _, tgt, d in list(H.out_edges(lower, data=True)):
                et = d.get("edge_type")
                if et != "hierarchy" and (upper, tgt, et) not in existing:
                    H.add_edge(upper, tgt, **d)
                    existing.add((upper, tgt, et))
                    added += 1
            for src, _, d in list(H.in_edges(lower, data=True)):
                et = d.get("edge_type")
                if et != "hierarchy" and (src, upper, et) not in existing:
                    H.add_edge(src, upper, **d)
                    existing.add((src, upper, et))
                    added += 1
        if added == 0:
            break
    return H



def remap_all_to_edge(G: nx.MultiDiGraph) -> nx.MultiDiGraph:
    """
    Type-agnostic: directional / correlational / moderation → 'edge'.
    Hierarchy edges keep their type (structurally distinct).
    Deduplicates after remapping.
    """
    H = nx.MultiDiGraph()
    for n, d in G.nodes(data=True): H.add_node(n, **d)
    seen = set()
    for u, v, d in G.edges(data=True):
        et = d.get("edge_type", "")
        new_et = et if et == "hierarchy" else "edge"
        key = (u, v, new_et)
        if key not in seen:
            H.add_edge(u, v, edge_type=new_et)
            seen.add(key)
    return H




def canonicalise_corr(G: nx.MultiDiGraph) -> nx.MultiDiGraph:
    """
    Make correlational edges direction-independent by normalising them
    to a canonical direction: min(u,v) → max(u,v) (lexicographic on
    the node's 'label' attribute, falling back to str(node_id)).

    Each undirected correlation becomes exactly ONE directed edge.
    Duplicate correlational edges between the same pair (e.g. A→B and B→A)
    are collapsed into one canonical edge.

    Non-correlational edges are left unchanged.
    """
    H = nx.MultiDiGraph()
    for n, d in G.nodes(data=True):
        H.add_node(n, **d)

    seen = set()  # (canonical_u, canonical_v, edge_type) for dedup

    def _label(n):
        return G.nodes[n].get("label", str(n))

    for u, v, d in G.edges(data=True):
        et = d.get("edge_type", "")
        if et == "correlational":
            # Canonical direction: sort by label, then by node id as tiebreaker
            lu, lv = _label(u), _label(v)
            if (lu, str(u)) > (lv, str(v)):
                cu, cv = v, u
            else:
                cu, cv = u, v
            key = (cu, cv, et)
            if key not in seen:
                nd = dict(d)
                H.add_edge(cu, cv, **nd)
                seen.add(key)
        else:
            key = (u, v, et)
            if key not in seen:
                H.add_edge(u, v, **d)
                seen.add(key)

    return H






def drop_corr_when_direction_exists(G: nx.MultiDiGraph) -> nx.MultiDiGraph:
    """Remove correlational edges when a directional edge exists on the same pair."""
    direction_pairs = {frozenset((u, v)) for u, v, d in G.edges(data=True)
                       if d.get("edge_type") == "directional"}
    if not direction_pairs:
        return G
    H = nx.MultiDiGraph()
    for n, d in G.nodes(data=True):
        H.add_node(n, **d)
    for u, v, d in G.edges(data=True):
        if d.get("edge_type") == "correlational" and frozenset((u, v)) in direction_pairs:
            continue
        H.add_edge(u, v, **d)
    return H


def merge_single_lv(G: nx.MultiDiGraph) -> nx.MultiDiGraph:
    """
    For each HV that has exactly ONE lower-level child LV (one hierarchy edge),
    merge them into a single node named 'HV (LV)':
      - Remove the hierarchy edge HV→LV
      - Rename both HV and LV to 'HV (LV)' (collapse into one node)
      - All edges incident to HV or LV are redirected to the merged node
      - Deduplicates after merging

    Applied to BOTH gt and pred as a preprocessing step.
    """
    # Find hierarchy edges and count children per HV
    hv_children = {}   # hv_node -> [lv_node, ...]
    for u, v, d in G.edges(data=True):
        if d.get("edge_type") == "hierarchy":
            hv_children.setdefault(u, []).append(v)

    # Build rename map for single-child HVs
    rename = {}   # old_node -> new_node_name
    drop_hier = set()  # (hv, lv) pairs to remove
    for hv, lvs in hv_children.items():
        if len(lvs) == 1:
            lv = lvs[0]
            hv_label = G.nodes[hv].get("label", str(hv))
            lv_label = G.nodes[lv].get("label", str(lv))
            new_name = f"{hv_label} ({lv_label})"
            rename[hv] = new_name
            rename[lv] = new_name
            drop_hier.add((hv, lv))

    if not rename:
        return G  # nothing to merge

    def mapped(n):
        return rename.get(n, n)

    H = nx.MultiDiGraph()
    seen = set()

    # Add nodes (merged nodes get the new label)
    for n, d in G.nodes(data=True):
        mn = mapped(n)
        if mn not in H:
            if n in rename:
                H.add_node(mn, label=mn)
            else:
                H.add_node(mn, **d)

    # Add edges (skip dropped hierarchy, dedup after rename)
    for u, v, d in G.edges(data=True):
        if d.get("edge_type") == "hierarchy" and (u, v) in drop_hier:
            continue  # drop this hierarchy edge
        mu, mv = mapped(u), mapped(v)
        if mu == mv:
            continue  # self-loop after merge
        et = d.get("edge_type", "")
        key = (mu, mv, et)
        if key not in seen:
            nd = dict(d)
            H.add_edge(mu, mv, **nd)
            seen.add(key)

    return H


print("Graph helpers defined (incl. canonicalise_corr, drop_corr_when_direction_exists, merge_single_lv).")


In [ ]:
# Cell 6 — Metric functions

# ── Basic stats ──
def graph_stats(G: nx.MultiDiGraph) -> dict:
    n  = G.number_of_nodes()
    m  = G.number_of_edges()
    et = Counter(d["edge_type"] for _,_,d in G.edges(data=True))
    return {
        "n_nodes": n, "n_edges": m,
        "n_hierarchy":     et.get("hierarchy", 0),
        "n_directional":   et.get("directional", 0),
        "n_correlational": et.get("correlational", 0),
        "n_moderation":    et.get("moderation", 0),
        "density": m / (n * (n-1)) if n > 1 else 0.0,
    }

# ── GED ──
def _rl(G): return nx.relabel_nodes(G, {n: i for i,n in enumerate(G.nodes())})
def _nsc(a,b): return 0
def _ndc(a):   return 1
def _nic(a):   return 1
def _esc(a,b): return 0 if a.get("edge_type") == b.get("edge_type") else 1
def _edc(a):   return 1
def _eic(a):   return 1

class _TimeoutError(Exception): pass
def _timeout_handler(s, f): raise _TimeoutError()

def run_ged(G_gt, G_pred, timeout=30.0):
    g1 = _rl(G_gt); g2 = _rl(G_pred)
    n1, n2 = g1.number_of_nodes(), g2.number_of_nodes()
    if n1 == 0 and n2 == 0: return 0.0, 0.0
    if n1 == 0: return float(n2 + g2.number_of_edges()), 1.0
    if n2 == 0: return float(n1 + g1.number_of_edges()), 1.0
    best = None
    old = signal.signal(signal.SIGALRM, _timeout_handler)
    signal.setitimer(signal.ITIMER_REAL, timeout)
    try:
        for d in nx.optimize_graph_edit_distance(
            g1, g2, node_subst_cost=_nsc, node_del_cost=_ndc, node_ins_cost=_nic,
            edge_subst_cost=_esc, edge_del_cost=_edc, edge_ins_cost=_eic):
            best = d
    except _TimeoutError: pass
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, old)
    if best is None:
        best = float(max(n1,n2) + max(g1.number_of_edges(), g2.number_of_edges()))
    denom = n1 + g1.number_of_edges() + n2 + g2.number_of_edges()
    return best, (best / denom if denom > 0 else 0.0)

# ── WL kernel ──
def wl_kernel(G1, G2, h=3):
    def _nl(G, n):
        it = Counter(G.edges[e].get("edge_type","?") for e in G.in_edges(n,  keys=True))
        ot = Counter(G.edges[e].get("edge_type","?") for e in G.out_edges(n, keys=True))
        return f"in={sorted(it.items())}_out={sorted(ot.items())}"
    l1 = {n: _nl(G1,n) for n in G1.nodes()}
    l2 = {n: _nl(G2,n) for n in G2.nodes()}
    h1 = Counter(l1.values()); h2 = Counter(l2.values())
    for _ in range(h):
        l1 = {n: f"{l1[n]}|{sorted([l1[nb] for nb in set(list(G1.predecessors(n))+list(G1.successors(n)))])}" for n in G1.nodes()}
        l2 = {n: f"{l2[n]}|{sorted([l2[nb] for nb in set(list(G2.predecessors(n))+list(G2.successors(n)))])}" for n in G2.nodes()}
        h1.update(l1.values()); h2.update(l2.values())
    al = set(h1) | set(h2)
    if not al: return 1.0 if (G1.number_of_nodes()==0 and G2.number_of_nodes()==0) else 0.0
    v1 = np.array([h1.get(l,0) for l in al], dtype=float)
    v2 = np.array([h2.get(l,0) for l in al], dtype=float)
    dot = np.dot(v1,v2); n1 = np.linalg.norm(v1); n2 = np.linalg.norm(v2)
    return dot/(n1*n2) if n1>0 and n2>0 else 0.0

# ── Spectral similarity ──
def _to_simple(G: nx.MultiDiGraph) -> nx.DiGraph:
    S = nx.DiGraph()
    for n, d in G.nodes(data=True): S.add_node(n, **d)
    for u, v, d in G.edges(data=True):
        if S.has_edge(u, v): S[u][v]["weight"] = S[u][v].get("weight",1) + 1
        else: S.add_edge(u, v, weight=1)
    return S

def spectral_similarity(G1, G2, max_eig=20):
    def _spec(G):
        n = G.number_of_nodes()
        if n == 0: return np.array([])
        A  = nx.to_numpy_array(G); As = (A + A.T) / 2.0
        D  = np.diag(As.sum(axis=1)); L = D - As
        dis = np.zeros_like(D); dv = np.diag(D); nz = dv > 1e-10
        dis[nz,nz] = 1.0 / np.sqrt(dv[nz])
        return np.sort(eigvalsh(dis @ L @ dis))[:max_eig]
    s1 = _spec(_to_simple(G1)); s2 = _spec(_to_simple(G2))
    ml = max(len(s1), len(s2))
    if ml == 0: return 1.0
    v1 = np.zeros(ml); v2 = np.zeros(ml)
    v1[:len(s1)] = s1; v2[:len(s2)] = s2
    dot = np.dot(v1,v2); n1 = np.linalg.norm(v1); n2 = np.linalg.norm(v2)
    return float(dot/(n1*n2)) if n1>0 and n2>0 else 0.0

# ── MCS-based edge F1 (branch-and-bound with greedy warm-start) ──────────────
#
# Finds the node mapping that maximises matched typed edges.
# Much faster than GED: uses a tight upper bound to prune the search tree,
# and a greedy solution to warm-start the best-known value.
# A 30s timeout is kept as a safety net for worst-case graphs (n≥15);
# the greedy result is always returned as a lower bound.

class _TimeoutError2(Exception): pass
def _timeout_handler2(s, f): raise _TimeoutError2()

def mcs_f1(G_gt, G_pred, timeout=120.0):
    m1 = G_gt.number_of_edges()
    m2 = G_pred.number_of_edges()
    if m1 == 0 and m2 == 0:
        return {"edge_matched":0,"edge_p":1.0,"edge_r":1.0,"edge_f1":1.0}
    if m1 == 0 or m2 == 0:
        return {"edge_matched":0,"edge_p":0.0,"edge_r":0.0,"edge_f1":0.0}

    # Order GT nodes by degree desc → high-degree nodes first finds good solutions early
    gt_nodes   = sorted(G_gt.nodes(), key=lambda n: G_gt.degree(n), reverse=True)
    pred_nodes = list(G_pred.nodes())

    pred_edge_set = {(u,v,d["edge_type"]) for u,v,d in G_pred.edges(data=True)}
    adj_out = {n:[(v,d["edge_type"]) for _,v,d in G_gt.out_edges(n,data=True)] for n in G_gt.nodes()}
    adj_in  = {n:[(u,d["edge_type"]) for u,_,d in G_gt.in_edges(n,data=True)]  for n in G_gt.nodes()}

    def count_em(gn, pn, phi):
        c = 0
        for v,t in adj_out.get(gn,[]):
            pv=phi.get(v)
            if pv and (pn,pv,t) in pred_edge_set: c+=1
        for u,t in adj_in.get(gn,[]):
            pu=phi.get(u)
            if pu and (pu,pn,t) in pred_edge_set: c+=1
        return c

    # Greedy warm-start: assign each GT node to the best available pred node
    def greedy():
        phi={}; avail=list(pred_nodes); em=0
        for gn in gt_nodes:
            best_c=-1; best_p=None
            for p in avail:
                c=count_em(gn,p,phi)
                if c>best_c: best_c=c; best_p=p
            if best_p is not None:
                phi[gn]=best_p; avail.remove(best_p); em+=best_c
        return em

    best = [greedy()]

    # Branch-and-bound
    def branch(depth, phi, avail, cur_em, unresolved):
        # Upper bound: cur_em + at most unresolved more matches
        if cur_em + unresolved <= best[0]: return
        if depth == len(gt_nodes):
            if cur_em > best[0]: best[0] = cur_em
            return
        gn = gt_nodes[depth]
        # Edges whose other endpoint is already assigned get resolved now
        resolved_now = (
            sum(1 for v,_ in adj_out.get(gn,[]) if phi.get(v))
          + sum(1 for u,_ in adj_in.get(gn,[])  if phi.get(u))
        )
        next_unresolved = unresolved - resolved_now
        # Try assigning to each available pred node
        for j, pn in enumerate(avail):
            new_em = cur_em + count_em(gn, pn, phi)
            phi[gn] = pn
            branch(depth+1, phi, avail[:j]+avail[j+1:], new_em, next_unresolved)
            del phi[gn]
        # Try deletion
        branch(depth+1, phi, avail, cur_em, next_unresolved)

    old_h = signal.signal(signal.SIGALRM, _timeout_handler2)
    signal.setitimer(signal.ITIMER_REAL, timeout)
    try:
        branch(0, {}, pred_nodes, 0, m1)
    except _TimeoutError2:
        pass
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, old_h)

    em=best[0]
    ep=em/m2 if m2>0 else (1.0 if m1==0 else 0.0)
    er=em/m1 if m1>0 else (1.0 if m2==0 else 0.0)
    ef=2*ep*er/(ep+er) if (ep+er)>0 else 0.0
    return {"edge_matched":em,"edge_p":ep,"edge_r":er,"edge_f1":ef}
print("Metric functions defined.")


In [ ]:
# Cell 7 — Evaluation runner and summary printer

def run_evaluation(gt_g, pred_g, ids):
    """Run all metrics for a set of graph pairs. Returns a per-article DataFrame."""
    import time
    stats_rows=[]; ged_rows=[]; wl_rows=[]; spec_rows=[]; f1_rows=[]
    n = len(ids)
    for i, aid in enumerate(ids):
        G_gt   = gt_g[aid]
        G_pred = pred_g[aid]
        n_gt   = G_gt.number_of_nodes();  m_gt   = G_gt.number_of_edges()
        n_pred = G_pred.number_of_nodes(); m_pred = G_pred.number_of_edges()
        print(f"  [{i+1:>3}/{n}] article {aid:>4} "
              f"| GT {n_gt}n/{m_gt}e  Pred {n_pred}n/{m_pred}e", end="", flush=True)
        t0 = time.time()

        gs = graph_stats(G_gt); ps = graph_stats(G_pred)
        row = {"article_id": aid}
        for k in gs:
            row[f"gt_{k}"]   = gs[k]
            row[f"pred_{k}"] = ps[k]
            if isinstance(gs[k], (int, float)):
                row[f"diff_{k}"] = ps[k] - gs[k]
        stats_rows.append(row)

        print("  GED", end="", flush=True)
        gr, gn = run_ged(G_gt, G_pred)
        ged_rows.append({"article_id": aid, "ged_raw": gr, "ged_normalized": gn})

        print("  WL", end="", flush=True)
        wl_rows.append({"article_id": aid, "wl_similarity": wl_kernel(G_gt, G_pred)})

        print("  Spec", end="", flush=True)
        spec_rows.append({"article_id": aid, "spectral_similarity": spectral_similarity(G_gt, G_pred)})

        print("  F1", end="", flush=True)
        r = mcs_f1(G_gt, G_pred); r["article_id"] = aid; f1_rows.append(r)

        elapsed = time.time() - t0
        print(f"  => F1={r['edge_f1']:.3f}  ({elapsed:.1f}s)")
    df = (pd.DataFrame(stats_rows)
            .merge(pd.DataFrame(ged_rows),  on="article_id")
            .merge(pd.DataFrame(wl_rows),   on="article_id")
            .merge(pd.DataFrame(spec_rows), on="article_id")
            .merge(pd.DataFrame(f1_rows),   on="article_id"))
    df["shd"]      = (df["gt_n_edges"] - df["edge_matched"]) + (df["pred_n_edges"] - df["edge_matched"])
    df["shd_norm"] = df["shd"] / df["gt_n_edges"].replace(0, float("nan"))
    return df


def print_condition_summary(df, label):
    """Print complete metrics for one evaluation condition."""
    W = 60
    sub  = df[df["gt_n_edges"] > 0]
    n_gt = (df["gt_n_edges"] > 0).sum()

    print("=" * W)
    print(f"  {label}")
    print("=" * W)
    print(f"  Articles (GT edges > 0)  : {n_gt} / {len(df)}")
    print(f"  Mean GT edges            : {sub['gt_n_edges'].mean():.2f}")
    print(f"  Mean Pred edges          : {sub['pred_n_edges'].mean():.2f}")
    print("-" * W)
    print(f"  GED normalized           : {sub['ged_normalized'].mean():.3f} +/- {sub['ged_normalized'].std():.3f}")
    print(f"  WL similarity            : {sub['wl_similarity'].mean():.4f} +/- {sub['wl_similarity'].std():.4f}")
    print(f"  Spectral similarity      : {sub['spectral_similarity'].mean():.4f} +/- {sub['spectral_similarity'].std():.4f}")
    print("-" * W)
    print(f"  Edge F1                  : P={sub['edge_p'].mean():.3f}  R={sub['edge_r'].mean():.3f}  F1={sub['edge_f1'].mean():.3f}")
    print(f"  SHD / |E_GT|             : {sub['shd_norm'].mean():.3f} +/- {sub['shd_norm'].std():.3f}")
    print("=" * W)

    # Per-article table
    print()
    print(f"  {'ID':>4}  {'GT':>4}  {'Pred':>4}  {'Match':>5}  {'SHD':>4}  {'SHD/GT':>7}  {'EdgeF1':>7}")
    print(f"  {'-'*4}  {'-'*4}  {'-'*4}  {'-'*5}  {'-'*4}  {'-'*7}  {'-'*7}")
    for _, row in df.iterrows():
        if row["gt_n_edges"] == 0 and row["pred_n_edges"] == 0:
            continue
        shd_s = f"{row['shd_norm']:.3f}" if not pd.isna(row["shd_norm"]) else "  N/A"
        print(f"  {int(row.article_id):>4}  {row.gt_n_edges:>4.0f}  {row.pred_n_edges:>4.0f}  "
              f"{row.edge_matched:>5.0f}  {row.shd:>4.0f}  {shd_s:>7}  {row.edge_f1:>7.3f}")
    print()


def save_condition(df, name):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    titles = {aid: gt_by_id[aid]["data"].get("TI","")[:80] for aid in overlap_ids}
    out = df.copy()
    out.insert(1, "title", out["article_id"].map(titles))
    path = os.path.join(OUTPUT_DIR, f"{name}.csv")
    out.to_csv(path, index=False)
    print(f"Saved {path}  ({out.shape[0]} rows x {out.shape[1]} cols)")



def print_before_after_comparison(df_pre, df_post, label):
    """Print a compact before/after table for one condition."""
    W = 68
    sub_pre  = df_pre[df_pre["gt_n_edges"]   > 0]
    sub_post = df_post[df_post["gt_n_edges"]  > 0]

    def _fmt(v, std=None):
        return f"{v:.3f}" + (f" ±{std:.3f}" if std is not None else "")

    rows = [
        ("Mean Pred edges",
         f"{sub_pre['pred_n_edges'].mean():.2f}",
         f"{sub_post['pred_n_edges'].mean():.2f}"),
        ("GED normalized",
         _fmt(sub_pre['ged_normalized'].mean(),  sub_pre['ged_normalized'].std()),
         _fmt(sub_post['ged_normalized'].mean(), sub_post['ged_normalized'].std())),
        ("WL similarity",
         _fmt(sub_pre['wl_similarity'].mean(),   sub_pre['wl_similarity'].std()),
         _fmt(sub_post['wl_similarity'].mean(),  sub_post['wl_similarity'].std())),
        ("Spectral similarity",
         _fmt(sub_pre['spectral_similarity'].mean(),  sub_pre['spectral_similarity'].std()),
         _fmt(sub_post['spectral_similarity'].mean(), sub_post['spectral_similarity'].std())),
        ("Edge P",
         f"{sub_pre['edge_p'].mean():.3f}",
         f"{sub_post['edge_p'].mean():.3f}"),
        ("Edge R",
         f"{sub_pre['edge_r'].mean():.3f}",
         f"{sub_post['edge_r'].mean():.3f}"),
        ("Edge F1",
         f"{sub_pre['edge_f1'].mean():.3f}",
         f"{sub_post['edge_f1'].mean():.3f}"),
        ("SHD / |E_GT|",
         _fmt(sub_pre['shd_norm'].mean(),  sub_pre['shd_norm'].std()),
         _fmt(sub_post['shd_norm'].mean(), sub_post['shd_norm'].std())),
    ]

    print("=" * W)
    print(f"  BEFORE vs AFTER step 5  |  {label}")
    print("=" * W)
    print(f"  {'Metric':<26} {'Before':>18}  {'After':>18}")
    print("-" * W)
    for name, pre_val, post_val in rows:
        print(f"  {name:<26} {pre_val:>18}  {post_val:>18}")
    print("=" * W)
    print()


print("All helpers defined.")

In [ ]:
# Cell 7.5 — Preprocessing applied to ALL graphs (GT + pred)
#
# 1. canonicalise_corr: normalise correlational edges to canonical direction.
# 2. merge_single_lv: for each HV with exactly one LV child, merge into 'HV (LV)'.
# 3. propagate_to_upper: copy LV edges onto HV for remaining hierarchies.
# 4. drop_corr_when_direction_exists: keep directional edges when both types exist.
#
# Order matters: merge first (removes trivial hierarchies), then propagate
# (for remaining multi-child hierarchies), and finally drop redundant correlations
# after propagation to catch edges introduced via hierarchy copying.

print("Preprocessing: canonicalise_corr → merge_single_lv → propagate_to_upper → drop_corr_when_direction_exists...")


def _apply_preprocessing(G: nx.MultiDiGraph) -> nx.MultiDiGraph:
    # 1) Canonicalise correlational direction early
    G = canonicalise_corr(G)
    # 2) Merge trivial HV->single LV hierarchies
    G = merge_single_lv(G)
    # 3) Propagate remaining lower-level relations to upper-level nodes
    G = propagate_to_upper(G)
    # 4) Canonicalise again because merge/propagation can re-introduce mirrored corr edges
    G = canonicalise_corr(G)
    # 5) If a directional edge exists for the pair, drop correlational duplicate
    return drop_corr_when_direction_exists(G)


for aid in overlap_ids:
    gt_graphs[aid] = _apply_preprocessing(gt_graphs[aid])
    pred_graphs_pre[aid] = _apply_preprocessing(pred_graphs_pre[aid])
    pred_graphs_post[aid] = _apply_preprocessing(pred_graphs_post[aid])


def _graph_summary_pp(graphs, label):
    print(f"\n{label}:")
    print(f"{'ID':>4} {'Nodes':>6} {'Edges':>6}  {'hier':>5} {'dir':>5} {'corr':>5} {'mod':>5}")
    for aid in overlap_ids:
        G = graphs[aid]
        et = Counter(d['edge_type'] for _, _, d in G.edges(data=True))
        print(f"{aid:>4} {G.number_of_nodes():>6} {G.number_of_edges():>6}  "
              f"{et.get('hierarchy', 0):>5} {et.get('directional', 0):>5} "
              f"{et.get('correlational', 0):>5} {et.get('moderation', 0):>5}")


_graph_summary_pp(gt_graphs, "GT graphs — after preprocessing")
_graph_summary_pp(pred_graphs_pre, "Pred PRE  — after preprocessing")
_graph_summary_pp(pred_graphs_post, "Pred POST — after preprocessing")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Condition A — All relationships  (hierarchy + dir + corr + mod)
# ════════════════════════════════════════════════════════════════

df_A_pre  = run_evaluation(gt_graphs, pred_graphs_pre,  overlap_ids)
print_condition_summary(df_A_pre,  "Condition A — All relationships  [BEFORE step 5]")
save_condition(df_A_pre,  "graph_eval_A_all_pre_140")


In [ ]:
# Thread-safe versions of timeout functions (for parallel processing)

def run_ged_no_timeout(G_gt, G_pred, max_iterations=1000):
    """GED without signal-based timeout (safer for parallel processing)."""
    g1 = _rl(G_gt); g2 = _rl(G_pred)
    n1, n2 = g1.number_of_nodes(), g2.number_of_nodes()
    if n1 == 0 and n2 == 0: return 0.0, 0.0
    if n1 == 0: return float(n2 + g2.number_of_edges()), 1.0
    if n2 == 0: return float(n1 + g1.number_of_edges()), 1.0
    
    best = None
    count = 0
    for d in nx.optimize_graph_edit_distance(
        g1, g2, node_subst_cost=_nsc, node_del_cost=_ndc, node_ins_cost=_nic,
        edge_subst_cost=_esc, edge_del_cost=_edc, edge_ins_cost=_eic):
        best = d
        count += 1
        if count >= max_iterations:
            break
    
    if best is None:
        return float('nan'), float('nan')
    norm = best / max(n1 + g1.number_of_edges(), n2 + g2.number_of_edges())
    return float(best), float(norm)

def mcs_f1_no_timeout(G_gt, G_pred, max_iterations=1000):
    """MCS F1 without signal-based timeout (safer for parallel processing)."""
    m1 = G_gt.number_of_edges()
    m2 = G_pred.number_of_edges()
    if m1 == 0 and m2 == 0:
        return {"edge_matched":0,"edge_p":1.0,"edge_r":1.0,"edge_f1":1.0}
    if m1 == 0 or m2 == 0:
        return {"edge_matched":0,"edge_p":0.0,"edge_r":0.0,"edge_f1":0.0}

    gt_nodes   = sorted(G_gt.nodes(), key=lambda n: G_gt.degree(n), reverse=True)
    pred_nodes = list(G_pred.nodes())
    pred_edge_set = {(u,v,d["edge_type"]) for u,v,d in G_pred.edges(data=True)}
    adj_out = {n:[(v,d["edge_type"]) for _,v,d in G_gt.out_edges(n,data=True)] for n in G_gt.nodes()}
    adj_in  = {n:[(u,d["edge_type"]) for u,_,d in G_gt.in_edges(n,data=True)]  for n in G_gt.nodes()}

    def _count_matched(mapping):
        c = 0
        for u_gt in G_gt.nodes():
            u_pred = mapping.get(u_gt)
            if u_pred is None: continue
            for v_gt, et in adj_out[u_gt]:
                v_pred = mapping.get(v_gt)
                if v_pred is not None and (u_pred, v_pred, et) in pred_edge_set:
                    c += 1
        return c

    def _greedy():
        mapping = {}
        remaining = set(pred_nodes)
        for u_gt in gt_nodes:
            if not remaining: break
            best_u, best_score = None, -1
            for u_pred in remaining:
                test = mapping.copy(); test[u_gt] = u_pred
                score = _count_matched(test)
                if score > best_score:
                    best_score, best_u = score, u_pred
            if best_u:
                mapping[u_gt] = best_u
                remaining.discard(best_u)
        return _count_matched(mapping)

    greedy_matched = _greedy()
    best_matched = greedy_matched
    
    def _bnb(idx, mapping, used, current_matched, iteration_count):
        nonlocal best_matched
        if iteration_count[0] >= max_iterations:
            return
        iteration_count[0] += 1
        
        if idx == len(gt_nodes):
            if current_matched > best_matched:
                best_matched = current_matched
            return
        
        rem = sum(G_gt.degree(gt_nodes[i]) for i in range(idx, len(gt_nodes)))
        if current_matched + rem <= best_matched:
            return
        
        u_gt = gt_nodes[idx]
        for u_pred in pred_nodes:
            if u_pred in used: continue
            mapping[u_gt] = u_pred
            used.add(u_pred)
            new_matched = current_matched
            for v_gt, et in adj_out[u_gt]:
                v_pred = mapping.get(v_gt)
                if v_pred is not None and (u_pred, v_pred, et) in pred_edge_set:
                    new_matched += 1
            for v_gt, et in adj_in[u_gt]:
                v_pred = mapping.get(v_gt)
                if v_pred is not None and (v_pred, u_pred, et) in pred_edge_set:
                    new_matched += 1
            _bnb(idx+1, mapping, used, new_matched, iteration_count)
            if iteration_count[0] >= max_iterations:
                return
            del mapping[u_gt]
            used.discard(u_pred)
        
        _bnb(idx+1, mapping, used, current_matched, iteration_count)

    iteration_count = [0]
    _bnb(0, {}, set(), 0, iteration_count)
    
    matched = best_matched
    prec = matched / m2 if m2 > 0 else 0.0
    rec  = matched / m1 if m1 > 0 else 0.0
    f1   = (2*prec*rec)/(prec+rec) if (prec+rec)>0 else 0.0
    return {"edge_matched": matched, "edge_p": prec, "edge_r": rec, "edge_f1": f1}

print("Thread-safe evaluation functions defined.")

In [ ]:
# Cell 7 — Evaluation runner and summary printer

def run_evaluation(gt_g, pred_g, ids):
    """Run all metrics for a set of graph pairs. Returns a per-article DataFrame."""
    import time
    stats_rows=[]; ged_rows=[]; wl_rows=[]; spec_rows=[]; f1_rows=[]
    n = len(ids)
    for i, aid in enumerate(ids):
        G_gt   = gt_g[aid]
        G_pred = pred_g[aid]
        n_gt   = G_gt.number_of_nodes();  m_gt   = G_gt.number_of_edges()
        n_pred = G_pred.number_of_nodes(); m_pred = G_pred.number_of_edges()
        print(f"  [{i+1:>3}/{n}] article {aid:>4} "
              f"| GT {n_gt}n/{m_gt}e  Pred {n_pred}n/{m_pred}e", end="", flush=True)
        t0 = time.time()

        gs = graph_stats(G_gt); ps = graph_stats(G_pred)
        row = {"article_id": aid}
        for k in gs:
            row[f"gt_{k}"]   = gs[k]
            row[f"pred_{k}"] = ps[k]
            if isinstance(gs[k], (int, float)):
                row[f"diff_{k}"] = ps[k] - gs[k]
        stats_rows.append(row)

        print("  GED", end="", flush=True)
        gr, gn = run_ged(G_gt, G_pred)
        ged_rows.append({"article_id": aid, "ged_raw": gr, "ged_normalized": gn})

        print("  WL", end="", flush=True)
        wl_rows.append({"article_id": aid, "wl_similarity": wl_kernel(G_gt, G_pred)})

        print("  Spec", end="", flush=True)
        spec_rows.append({"article_id": aid, "spectral_similarity": spectral_similarity(G_gt, G_pred)})

        print("  F1", end="", flush=True)
        r = mcs_f1(G_gt, G_pred); r["article_id"] = aid; f1_rows.append(r)

        elapsed = time.time() - t0
        print(f"  => F1={r['edge_f1']:.3f}  ({elapsed:.1f}s)")
    df = (pd.DataFrame(stats_rows)
            .merge(pd.DataFrame(ged_rows),  on="article_id")
            .merge(pd.DataFrame(wl_rows),   on="article_id")
            .merge(pd.DataFrame(spec_rows), on="article_id")
            .merge(pd.DataFrame(f1_rows),   on="article_id"))
    df["shd"]      = (df["gt_n_edges"] - df["edge_matched"]) + (df["pred_n_edges"] - df["edge_matched"])
    df["shd_norm"] = df["shd"] / df["gt_n_edges"].replace(0, float("nan"))
    return df


def print_condition_summary(df, label):
    """Print complete metrics for one evaluation condition."""
    W = 60
    sub  = df[df["gt_n_edges"] > 0]
    n_gt = (df["gt_n_edges"] > 0).sum()

    print("=" * W)
    print(f"  {label}")
    print("=" * W)
    print(f"  Articles (GT edges > 0)  : {n_gt} / {len(df)}")
    print(f"  Mean GT edges            : {sub['gt_n_edges'].mean():.2f}")
    print(f"  Mean Pred edges          : {sub['pred_n_edges'].mean():.2f}")
    print("-" * W)
    print(f"  GED normalized           : {sub['ged_normalized'].mean():.3f} +/- {sub['ged_normalized'].std():.3f}")
    print(f"  WL similarity            : {sub['wl_similarity'].mean():.4f} +/- {sub['wl_similarity'].std():.4f}")
    print(f"  Spectral similarity      : {sub['spectral_similarity'].mean():.4f} +/- {sub['spectral_similarity'].std():.4f}")
    print("-" * W)
    print(f"  Edge F1                  : P={sub['edge_p'].mean():.3f}  R={sub['edge_r'].mean():.3f}  F1={sub['edge_f1'].mean():.3f}")
    print(f"  SHD / |E_GT|             : {sub['shd_norm'].mean():.3f} +/- {sub['shd_norm'].std():.3f}")
    print("=" * W)

    # Per-article table
    print()
    print(f"  {'ID':>4}  {'GT':>4}  {'Pred':>4}  {'Match':>5}  {'SHD':>4}  {'SHD/GT':>7}  {'EdgeF1':>7}")
    print(f"  {'-'*4}  {'-'*4}  {'-'*4}  {'-'*5}  {'-'*4}  {'-'*7}  {'-'*7}")
    for _, row in df.iterrows():
        if row["gt_n_edges"] == 0 and row["pred_n_edges"] == 0:
            continue
        shd_s = f"{row['shd_norm']:.3f}" if not pd.isna(row["shd_norm"]) else "  N/A"
        print(f"  {int(row.article_id):>4}  {row.gt_n_edges:>4.0f}  {row.pred_n_edges:>4.0f}  "
              f"{row.edge_matched:>5.0f}  {row.shd:>4.0f}  {shd_s:>7}  {row.edge_f1:>7.3f}")
    print()


def save_condition(df, name):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    titles = {aid: gt_by_id[aid]["data"].get("TI","")[:80] for aid in overlap_ids}
    out = df.copy()
    out.insert(1, "title", out["article_id"].map(titles))
    path = os.path.join(OUTPUT_DIR, f"{name}.csv")
    out.to_csv(path, index=False)
    print(f"Saved {path}  ({out.shape[0]} rows x {out.shape[1]} cols)")



def print_before_after_comparison(df_pre, df_post, label):
    """Print a compact before/after table for one condition."""
    W = 68
    sub_pre  = df_pre[df_pre["gt_n_edges"]   > 0]
    sub_post = df_post[df_post["gt_n_edges"]  > 0]

    def _fmt(v, std=None):
        return f"{v:.3f}" + (f" ±{std:.3f}" if std is not None else "")

    rows = [
        ("Mean Pred edges",
         f"{sub_pre['pred_n_edges'].mean():.2f}",
         f"{sub_post['pred_n_edges'].mean():.2f}"),
        ("GED normalized",
         _fmt(sub_pre['ged_normalized'].mean(),  sub_pre['ged_normalized'].std()),
         _fmt(sub_post['ged_normalized'].mean(), sub_post['ged_normalized'].std())),
        ("WL similarity",
         _fmt(sub_pre['wl_similarity'].mean(),   sub_pre['wl_similarity'].std()),
         _fmt(sub_post['wl_similarity'].mean(),  sub_post['wl_similarity'].std())),
        ("Spectral similarity",
         _fmt(sub_pre['spectral_similarity'].mean(),  sub_pre['spectral_similarity'].std()),
         _fmt(sub_post['spectral_similarity'].mean(), sub_post['spectral_similarity'].std())),
        ("Edge P",
         f"{sub_pre['edge_p'].mean():.3f}",
         f"{sub_post['edge_p'].mean():.3f}"),
        ("Edge R",
         f"{sub_pre['edge_r'].mean():.3f}",
         f"{sub_post['edge_r'].mean():.3f}"),
        ("Edge F1",
         f"{sub_pre['edge_f1'].mean():.3f}",
         f"{sub_post['edge_f1'].mean():.3f}"),
        ("SHD / |E_GT|",
         _fmt(sub_pre['shd_norm'].mean(),  sub_pre['shd_norm'].std()),
         _fmt(sub_post['shd_norm'].mean(), sub_post['shd_norm'].std())),
    ]

    print("=" * W)
    print(f"  BEFORE vs AFTER step 5  |  {label}")
    print("=" * W)
    print(f"  {'Metric':<26} {'Before':>18}  {'After':>18}")
    print("-" * W)
    for name, pre_val, post_val in rows:
        print(f"  {name:<26} {pre_val:>18}  {post_val:>18}")
    print("=" * W)
    print()


print("All helpers defined.")


In [ ]:
# Cell — Parallel evaluation runner (uses thread-safe functions)

def run_evaluation_parallel(gt_g, pred_g, ids, n_jobs=8):
    """Run all metrics in parallel using joblib with thread-safe functions.
    
    Args:
        gt_g: dict of ground truth graphs
        pred_g: dict of predicted graphs
        ids: list of article IDs to process
        n_jobs: number of parallel jobs (-1 = all CPUs)
    """
    import time
    from joblib import Parallel, delayed
    
    def process_single_article(aid):
        """Process a single article - uses no-timeout versions."""
        G_gt = gt_g[aid]
        G_pred = pred_g[aid]
        
        result = {"article_id": aid}
        
        # Graph stats
        gs = graph_stats(G_gt)
        ps = graph_stats(G_pred)
        for k in gs:
            result[f"gt_{k}"] = gs[k]
            result[f"pred_{k}"] = ps[k]
            if isinstance(gs[k], (int, float)):
                result[f"diff_{k}"] = ps[k] - gs[k]
        
        # GED - use thread-safe version
        gr, gn = run_ged_no_timeout(G_gt, G_pred)
        result["ged_raw"] = gr
        result["ged_normalized"] = gn
        
        # WL kernel
        result["wl_similarity"] = wl_kernel(G_gt, G_pred)
        
        # Spectral similarity
        result["spectral_similarity"] = spectral_similarity(G_gt, G_pred)
        
        # F1 metrics - use thread-safe version
        f1_result = mcs_f1_no_timeout(G_gt, G_pred)
        result.update(f1_result)
        
        return result
    
    print(f"Starting parallel evaluation with {n_jobs if n_jobs > 0 else 'all'} workers...")
    t_start = time.time()
    
    # Run in parallel
    results = Parallel(n_jobs=n_jobs, backend='loky', verbose=5)(
        delayed(process_single_article)(aid) for aid in ids
    )
    
    # Build DataFrame
    df = pd.DataFrame(results)
    df["shd"] = (df["gt_n_edges"] - df["edge_matched"]) + (df["pred_n_edges"] - df["edge_matched"])
    df["shd_norm"] = df["shd"] / df["gt_n_edges"].replace(0, float("nan"))
    
    elapsed = time.time() - t_start
    n = len(results)
    print(f"\nCompleted {n} articles in {elapsed:.1f}s ({elapsed/n:.2f}s per article)")
    
    return df

print("Parallel evaluation function defined.")

In [ ]:
# Cell 7.5 — Preprocessing applied to ALL graphs (GT + pred)
#
# 1. canonicalise_corr: normalise correlational edges to canonical direction.
# 2. merge_single_lv: for each HV with exactly one LV child, merge into 'HV (LV)'.
# 3. propagate_to_upper: copy LV edges onto HV for remaining hierarchies.
# 4. drop_corr_when_direction_exists: keep directional edges when both types exist.
#
# Order matters: merge first (removes trivial hierarchies), then propagate
# (for remaining multi-child hierarchies), and finally drop redundant correlations
# after propagation to catch edges introduced via hierarchy copying.

print("Preprocessing: canonicalise_corr → merge_single_lv → propagate_to_upper → drop_corr_when_direction_exists...")


def _apply_preprocessing(G: nx.MultiDiGraph) -> nx.MultiDiGraph:
    # 1) Canonicalise correlational direction early
    G = canonicalise_corr(G)
    # 2) Merge trivial HV->single LV hierarchies
    G = merge_single_lv(G)
    # 3) Propagate remaining lower-level relations to upper-level nodes
    G = propagate_to_upper(G)
    # 4) Canonicalise again because merge/propagation can re-introduce mirrored corr edges
    G = canonicalise_corr(G)
    # 5) If a directional edge exists for the pair, drop correlational duplicate
    return drop_corr_when_direction_exists(G)


for aid in overlap_ids:
    gt_graphs[aid] = _apply_preprocessing(gt_graphs[aid])
    pred_graphs_pre[aid] = _apply_preprocessing(pred_graphs_pre[aid])
    pred_graphs_post[aid] = _apply_preprocessing(pred_graphs_post[aid])


def _graph_summary_pp(graphs, label):
    print(f"\n{label}:")
    print(f"{'ID':>4} {'Nodes':>6} {'Edges':>6}  {'hier':>5} {'dir':>5} {'corr':>5} {'mod':>5}")
    for aid in overlap_ids:
        G = graphs[aid]
        et = Counter(d['edge_type'] for _, _, d in G.edges(data=True))
        print(f"{aid:>4} {G.number_of_nodes():>6} {G.number_of_edges():>6}  "
              f"{et.get('hierarchy', 0):>5} {et.get('directional', 0):>5} "
              f"{et.get('correlational', 0):>5} {et.get('moderation', 0):>5}")


_graph_summary_pp(gt_graphs, "GT graphs — after preprocessing")
_graph_summary_pp(pred_graphs_pre, "Pred PRE  — after preprocessing")
_graph_summary_pp(pred_graphs_post, "Pred POST — after preprocessing")


In [ ]:
# ════════════════════════════════════════════════════════════════
# Condition A — All relationships  (hierarchy + dir + corr + mod)
# ════════════════════════════════════════════════════════════════

df_A_pre  = run_evaluation_parallel(gt_graphs, pred_graphs_pre,  overlap_ids)
#df_A_post = run_evaluation(gt_graphs, pred_graphs_post, overlap_ids)

print_condition_summary(df_A_pre,  "Condition A — All relationships  [BEFORE step 5]")
#print_condition_summary(df_A_post, "Condition A — All relationships  [AFTER  step 5]")
#print_before_after_comparison(df_A_pre, "Condition A — All relationships")

save_condition(df_A_pre,  "graph_eval_A_per_type_pre_newstep4")
#save_condition(df_A_post, "graph_eval_A_all_post")


# ── Per-edge-type F1 breakdown for Condition A ──────────────────────────────
# For each edge type, filter both GT and pred to only that type, then run mcs_f1.
# This shows how well the pipeline performs on each relationship type separately.

_EDGE_TYPES = ["hierarchy", "directional", "correlational", "moderation"]


def per_type_f1(gt_g, pred_g, ids):
    """Compute mcs_f1 per edge type. Returns a DataFrame."""
    rows = []
    for et in _EDGE_TYPES:
        gt_filt   = {aid: keep_only(gt_g[aid],   (et,)) for aid in ids}
        pred_filt = {aid: keep_only(pred_g[aid],  (et,)) for aid in ids}
        for aid in ids:
            G_gt   = gt_filt[aid]
            G_pred = pred_filt[aid]
            m1 = G_gt.number_of_edges()
            m2 = G_pred.number_of_edges()
            r  = mcs_f1(G_gt, G_pred)
            r.update({"article_id": aid, "edge_type": et,
                      "gt_edges": m1, "pred_edges": m2})
            rows.append(r)
    return pd.DataFrame(rows)


def print_per_type_summary(df_pt, label):
    """Print aggregate per-type F1."""
    W = 68
    print("=" * W)
    print(f"  Per-edge-type F1  |  {label}")
    print("=" * W)
    print(f"  {'Type':<16} {'GT_e':>5} {'Pred_e':>6} {'Match':>6}  "
          f"{'P':>6} {'R':>6} {'F1':>6}  {'n_art':>5}")
    print("-" * W)
    for et in _EDGE_TYPES:
        sub = df_pt[df_pt["edge_type"] == et]
        # Only articles where at least GT or pred has edges of this type
        active = sub[(sub["gt_edges"] > 0) | (sub["pred_edges"] > 0)]
        if len(active) == 0:
            print(f"  {et:<16} {'—':>5} {'—':>6} {'—':>6}  {'—':>6} {'—':>6} {'—':>6}  {0:>5}")
            continue
        print(f"  {et:<16} "
              f"{active['gt_edges'].sum():>5.0f} "
              f"{active['pred_edges'].sum():>6.0f} "
              f"{active['edge_matched'].sum():>6.0f}  "
              f"{active['edge_p'].mean():>6.3f} "
              f"{active['edge_r'].mean():>6.3f} "
              f"{active['edge_f1'].mean():>6.3f}  "
              f"{len(active):>5}")
    # Overall
    all_active = df_pt[(df_pt["gt_edges"] > 0) | (df_pt["pred_edges"] > 0)]
    print("-" * W)
    print(f"  {'ALL':<16} "
          f"{all_active['gt_edges'].sum():>5.0f} "
          f"{all_active['pred_edges'].sum():>6.0f} "
          f"{all_active['edge_matched'].sum():>6.0f}  "
          f"{all_active['edge_p'].mean():>6.3f} "
          f"{all_active['edge_r'].mean():>6.3f} "
          f"{all_active['edge_f1'].mean():>6.3f}  "
          f"{len(all_active):>5}")
    print("=" * W)
    print()


print("\nComputing per-edge-type F1 for Condition A...")
print("[BEFORE validation]")
df_A_pt_pre  = per_type_f1(gt_graphs, pred_graphs_pre,  overlap_ids)
print_per_type_summary(df_A_pt_pre,  "Condition A — BEFORE validation")

print("[AFTER validation]")
df_A_pt_post = per_type_f1(gt_graphs, pred_graphs_post, overlap_ids)
print_per_type_summary(df_A_pt_post, "Condition A — AFTER  validation")

# Save
df_A_pt_pre.to_csv(os.path.join(OUTPUT_DIR,  "graph_eval_A_per_type_pre_330.csv"),  index=False)
#df_A_pt_post.to_csv(os.path.join(OUTPUT_DIR, "graph_eval_A_per_type_post.csv"), index=False)
print(f"Saved per-type F1 to {OUTPUT_DIR}")


In [ ]:
save_condition(df_A_pre,  "graph_eval_A_all_pre")


In [ ]:
# ════════════════════════════════════════════════════════════════
# Condition B — Merged corr+dir  (correlational treated as directional)
# ════════════════════════════════════════════════════════════════

gt_B      = {aid: remap_corr_to_dir(gt_graphs[aid])        for aid in overlap_ids}
pred_B_pre  = {aid: remap_corr_to_dir(pred_graphs_pre[aid])  for aid in overlap_ids}
pred_B_post = {aid: remap_corr_to_dir(pred_graphs_post[aid]) for aid in overlap_ids}

df_B_pre  = run_evaluation(gt_B, pred_B_pre,  overlap_ids)
df_B_post = run_evaluation(gt_B, pred_B_post, overlap_ids)

print_condition_summary(df_B_pre,  "Condition B — Merged corr+dir  [BEFORE step 5]")
print_condition_summary(df_B_post, "Condition B — Merged corr+dir  [AFTER  step 5]")
print_before_after_comparison(df_B_pre, df_B_post, "Condition B — Merged corr+dir")

save_condition(df_B_pre,  "graph_eval_B_merged_pre")
save_condition(df_B_post, "graph_eval_B_merged_post")


In [ ]:
# ════════════════════════════════════════════════════════════════
# Condition C — Without hierarchy  (dir + corr + mod only)
# ════════════════════════════════════════════════════════════════

gt_C        = {aid: filter_graph(gt_graphs[aid],         exclude_types=("hierarchy",)) for aid in overlap_ids}
pred_C_pre  = {aid: filter_graph(pred_graphs_pre[aid],   exclude_types=("hierarchy",)) for aid in overlap_ids}
pred_C_post = {aid: filter_graph(pred_graphs_post[aid],  exclude_types=("hierarchy",)) for aid in overlap_ids}

df_C_pre  = run_evaluation(gt_C, pred_C_pre,  overlap_ids)
df_C_post = run_evaluation(gt_C, pred_C_post, overlap_ids)

print_condition_summary(df_C_pre,  "Condition C — Without hierarchy  [BEFORE step 5]")
print_condition_summary(df_C_post, "Condition C — Without hierarchy  [AFTER  step 5]")
print_before_after_comparison(df_C_pre, df_C_post, "Condition C — Without hierarchy")

save_condition(df_C_pre,  "graph_eval_C_no_hier_pre")
save_condition(df_C_post, "graph_eval_C_no_hier_post")


In [ ]:
# ════════════════════════════════════════════════════════════════
# Condition D — Hierarchy only
# ════════════════════════════════════════════════════════════════

gt_D        = {aid: keep_only(gt_graphs[aid],        ("hierarchy",)) for aid in overlap_ids}
pred_D_pre  = {aid: keep_only(pred_graphs_pre[aid],  ("hierarchy",)) for aid in overlap_ids}
pred_D_post = {aid: keep_only(pred_graphs_post[aid], ("hierarchy",)) for aid in overlap_ids}

df_D_pre  = run_evaluation(gt_D, pred_D_pre,  overlap_ids)
df_D_post = run_evaluation(gt_D, pred_D_post, overlap_ids)

print_condition_summary(df_D_pre,  "Condition D — Hierarchy only  [BEFORE step 5]")
print_condition_summary(df_D_post, "Condition D — Hierarchy only  [AFTER  step 5]")
print_before_after_comparison(df_D_pre, df_D_post, "Condition D — Hierarchy only")

save_condition(df_D_pre,  "graph_eval_D_hier_only_pre")
save_condition(df_D_post, "graph_eval_D_hier_only_post")


In [ ]:
# ════════════════════════════════════════════════════════════════
# Condition E — Discard hierarchy lower nodes
#
# Lower nodes = targets of hierarchy edges.
# Both the lower nodes and all their incident edges are removed.
# This evaluates the abstract-level graph: upper hierarchy nodes +
# all directional / correlational / moderation relationships
# among the remaining nodes.
# ════════════════════════════════════════════════════════════════

gt_E        = {aid: discard_hierarchy_lower_nodes(gt_graphs[aid])        for aid in overlap_ids}
pred_E_pre  = {aid: discard_hierarchy_lower_nodes(pred_graphs_pre[aid])  for aid in overlap_ids}
pred_E_post = {aid: discard_hierarchy_lower_nodes(pred_graphs_post[aid]) for aid in overlap_ids}

df_E_pre  = run_evaluation(gt_E, pred_E_pre,  overlap_ids)
df_E_post = run_evaluation(gt_E, pred_E_post, overlap_ids)

print_condition_summary(df_E_pre,  "Condition E — Discard hierarchy lower nodes  [BEFORE step 5]")
print_condition_summary(df_E_post, "Condition E — Discard hierarchy lower nodes  [AFTER  step 5]")
print_before_after_comparison(df_E_pre, df_E_post, "Condition E — Discard hierarchy lower nodes")

save_condition(df_E_pre,  "graph_eval_E_no_lower_pre")
save_condition(df_E_post, "graph_eval_E_no_lower_post")


In [ ]:
# ════════════════════════════════════════════════════════════════
# Condition F — Type-agnostic (dir/corr/mod → 'edge'; hierarchy kept)
#
# Applied to BOTH gt and pred graphs.
# This evaluates whether the pipeline finds the right variable pairs
# regardless of how it labels the relationship type.
# ════════════════════════════════════════════════════════════════

gt_F        = {aid: remap_all_to_edge(gt_graphs[aid])        for aid in overlap_ids}
pred_F_pre  = {aid: remap_all_to_edge(pred_graphs_pre[aid])  for aid in overlap_ids}
pred_F_post = {aid: remap_all_to_edge(pred_graphs_post[aid]) for aid in overlap_ids}

df_F_pre  = run_evaluation(gt_F, pred_F_pre,  overlap_ids)
df_F_post = run_evaluation(gt_F, pred_F_post, overlap_ids)

print_condition_summary(df_F_pre,  "Condition F — Type-agnostic  [BEFORE validation]")
print_condition_summary(df_F_post, "Condition F — Type-agnostic  [AFTER  validation]")
print_before_after_comparison(df_F_pre, df_F_post, "Condition F — Type-agnostic")

save_condition(df_F_pre,  "graph_eval_F_type_agnostic_pre")
save_condition(df_F_post, "graph_eval_F_type_agnostic_post")


In [ ]:
# ════════════════════════════════════════════════════════════════
# Condition G — Type-agnostic + upper hierarchy only
#
# 1. Discard lower-level nodes (targets of hierarchy edges)
# 2. Remap remaining dir/corr/mod edges to 'edge' (type-agnostic)
#
# This evaluates the abstract-level variable graph: do we find
# the right higher-level variable pairs, regardless of edge type?
# Applied to BOTH gt and pred.
# ════════════════════════════════════════════════════════════════

def _upper_type_agnostic(G):
    return remap_all_to_edge(discard_hierarchy_lower_nodes(G))

gt_G        = {aid: _upper_type_agnostic(gt_graphs[aid])        for aid in overlap_ids}
pred_G_pre  = {aid: _upper_type_agnostic(pred_graphs_pre[aid])  for aid in overlap_ids}
pred_G_post = {aid: _upper_type_agnostic(pred_graphs_post[aid]) for aid in overlap_ids}

df_G_pre  = run_evaluation(gt_G, pred_G_pre,  overlap_ids)
df_G_post = run_evaluation(gt_G, pred_G_post, overlap_ids)

print_condition_summary(df_G_pre,  "Condition G — Type-agnostic + upper only  [BEFORE validation]")
print_condition_summary(df_G_post, "Condition G — Type-agnostic + upper only  [AFTER  validation]")
print_before_after_comparison(df_G_pre, df_G_post, "Condition G — Type-agnostic + upper only")

save_condition(df_G_pre,  "graph_eval_G_agnostic_upper_pre")
save_condition(df_G_post, "graph_eval_G_agnostic_upper_post")


In [ ]:
# Cell 12 — Comparison CSV
#
# Columns: abstract | gt_variables | pred_variables
#          | gt_hierarchy  | pred_hierarchy_pre  | pred_hierarchy_post
#          | gt_directional| pred_directional_pre| pred_directional_post
#          | gt_correlational | pred_correlational_pre | pred_correlational_post
#          | gt_moderation | pred_moderation_pre | pred_moderation_post

# ── helpers to render GT graph edges as readable strings ─────────────────────

def _node_label(G, n):
    return G.nodes[n].get("label", str(n))

def gt_variables_str(G):
    return "; ".join(sorted(G.nodes[n].get("label", n) for n in G.nodes()))

def gt_edges_str(G, edge_type):
    """Return semicolon-separated edge strings for one edge type."""
    out = []
    if edge_type == "moderation":
        # group by moderator node → "mod moderates (v1, v2)"
        from collections import defaultdict
        groups = defaultdict(list)
        for u, v, d in G.edges(data=True):
            if d.get("edge_type") == "moderation":
                groups[u].append(_node_label(G, v))
        for mod_node, targets in groups.items():
            mod_lbl = _node_label(G, mod_node)
            out.append(f"{mod_lbl} moderates ({', '.join(sorted(targets))})")
    elif edge_type == "correlational":
        for u, v, d in G.edges(data=True):
            if d.get("edge_type") == edge_type:
                out.append(f"{_node_label(G, u)} <-> {_node_label(G, v)}")
    else:
        for u, v, d in G.edges(data=True):
            if d.get("edge_type") == edge_type:
                out.append(f"{_node_label(G, u)} -> {_node_label(G, v)}")
    return "; ".join(out) if out else ""

# ── build the comparison dataframe ───────────────────────────────────────────

rows = []
for aid in overlap_ids:
    G_gt   = gt_graphs[aid]
    p_row  = pred_df[pred_df["num_id"] == aid].iloc[0]
    title  = gt_by_id[aid]["data"].get("TI", "")
    abstract = gt_by_id[aid]["data"].get("AB", "")

    def _col(col):
        v = p_row.get(col, "")
        return "" if pd.isna(v) else str(v).strip()

    rows.append({
        "article_id":               aid,
        "title":                    title,
        "abstract":                 abstract,
        # variables
        "gt_variables":             gt_variables_str(G_gt),
        "pred_variables":           _col("step2_final_vars"),
        # hierarchy
        "gt_hierarchy":             gt_edges_str(G_gt, "hierarchy"),
        "pred_hierarchy_pre":       _col("step2_hierarchy"),
        "pred_hierarchy_post":      _col("step5_2_hierarchy"),
        # directional
        "gt_directional":           gt_edges_str(G_gt, "directional"),
        "pred_directional_pre":     _col("step4_directional"),
        "pred_directional_post":    _col("step5_2_directional"),
        # correlational
        "gt_correlational":         gt_edges_str(G_gt, "correlational"),
        "pred_correlational_pre":   _col("step4_correlational"),
        "pred_correlational_post":  _col("step5_2_correlational"),
        # moderation
        "gt_moderation":            gt_edges_str(G_gt, "moderation"),
        "pred_moderation_pre":      _col("step4_moderation"),
        "pred_moderation_post":     _col("step5_2_moderation"),
    })

df_compare = pd.DataFrame(rows)

# ── save ─────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
out_path = os.path.join(OUTPUT_DIR, "comparison.csv")
df_compare.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Saved {out_path}  ({df_compare.shape[0]} rows x {df_compare.shape[1]} cols)")
print(f"Columns: {list(df_compare.columns)}")
